# Step 4c — Extended brain-immune ssGSEA modules

**Motivation.** Reflect Grabowski 2021 (*J Neurooncol*) and Sampson 2020 (*Nat Rev Cancer*) review concepts directly in the ecotype framework.

**Modules added (12 + 1 derived):**

- **A1**: `Exhaustion_core`, `Senescence_T_cell`, `TCM_Naive_loss`
- **A2**: `MDSC_granulocytic`, `MDSC_monocytic`
- **A3**: `Hypoxia_HIF_targets`
- **A4**: `Antigen_presentation_HLA_I`, `Antigen_presentation_HLA_II`, `HLA_I_minus_II` (derived)
- **A6**: `CSF1R_TAM_axis`, `CCR2_chemokine_axis`
- **Extra**: `Treg_signature`, `Suppressive_cytokine_axis`

**Inputs** — `output/tpm_for_cibersortx.tsv` (gene × sample, log-TPM source); `output/ecotype_LM22_main_k3_annotated.tsv`; `data/fecci_sampson_extended_modules.gmt`.

**Outputs** — `output/step4c_ssGSEA_scores.tsv`, `step4c_KW_by_ecotype.tsv`, `step4c_Dunn_posthoc.tsv`, `step4c_KW_by_cohort.tsv`, `step4c_module_summary.json`, `output/figs_step4c/*.png|pdf` (300 dpi).


In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd
import gseapy as gp
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ROOT     = Path("/sessions/blissful-gifted-dirac/mnt/Open PBTA")
OUT      = ROOT / "output"
FIGDIR   = OUT / "figs_step4c"; FIGDIR.mkdir(exist_ok=True, parents=True)
TPM_FN   = OUT / "tpm_for_cibersortx.tsv"
ECO_FN   = OUT / "ecotype_LM22_main_k3_annotated.tsv"
META_FN  = OUT / "ecotype_assignment_k3_annotated.tsv"
GMT_FN   = ROOT / "data" / "fecci_sampson_extended_modules.gmt"

ECOTYPE_ORDER = ["Lymphocyte-inflamed", "Myeloid-dominant", "Immune-desert"]
ECOTYPE_COLORS = {"Lymphocyte-inflamed":"#2C7BB6","Myeloid-dominant":"#D7301F","Immune-desert":"#7F7F7F"}
COHORT_ORDER  = ["DMG_K27","DHG_G34","pHGG_WT","IHG"]
COHORT_COLORS = {"DMG_K27":"#1B9E77","DHG_G34":"#D95F02","pHGG_WT":"#7570B3","IHG":"#E7298A"}
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"font.size":9,
                     "axes.spines.top":False,"axes.spines.right":False,"pdf.fonttype":42})

## 1. Load TPM + ecotype assignment + gene-set GMT

In [2]:
tpm = pd.read_csv(TPM_FN, sep="\t")
tpm = tpm.rename(columns={tpm.columns[0]: "GeneSymbol"}).drop_duplicates("GeneSymbol").set_index("GeneSymbol")
eco  = pd.read_csv(ECO_FN, sep="\t").set_index("Kids_First_Biospecimen_ID")
meta = pd.read_csv(META_FN).set_index("Kids_First_Biospecimen_ID")
eco  = eco[["ecotype"]].join(meta[["cohort_group","location_class","age_dev_group"]], how="left")
keep = [b for b in tpm.columns if b in eco.index]
tpm, eco = tpm[keep], eco.loc[keep]
print("TPM matrix:", tpm.shape, "| ecotype labels:", eco.ecotype.value_counts().to_dict())

def load_gmt(fn):
    out = {}
    for line in open(fn):
        n, _, *g = line.rstrip("\n").split("\t")
        out[n] = [x for x in g if x]
    return out
sets = load_gmt(GMT_FN)
for n, gs in sets.items():
    print(f"  {n}: {sum(g in tpm.index for g in gs)}/{len(gs)} genes present")

TPM matrix: (55408, 349) | ecotype labels: {'Myeloid-dominant': 160, 'Lymphocyte-inflamed': 111, 'Immune-desert': 78}
  Exhaustion_core: 12/12 genes present
  Senescence_T_cell: 8/9 genes present
  TCM_Naive_loss: 8/8 genes present
  MDSC_granulocytic: 12/12 genes present
  MDSC_monocytic: 11/11 genes present
  Hypoxia_HIF_targets: 13/13 genes present
  Antigen_presentation_HLA_I: 12/12 genes present
  Antigen_presentation_HLA_II: 10/10 genes present
  CSF1R_TAM_axis: 7/7 genes present
  CCR2_chemokine_axis: 7/7 genes present
  Treg_signature: 10/10 genes present
  Suppressive_cytokine_axis: 10/10 genes present


## 2. Run ssGSEA (gseapy, rank-normalised) on log2(TPM+1)

In [3]:
log_tpm = np.log2(tpm.astype(float) + 1.0)
res = gp.ssgsea(data=log_tpm, gene_sets=sets, sample_norm_method="rank",
                no_plot=True, threads=1, min_size=3, max_size=500,
                permutation_num=0, outdir=None)
scores = res.res2d.copy()
scores["NES"] = pd.to_numeric(scores["NES"], errors="coerce")
scores = scores.pivot(index="Name", columns="Term", values="NES").astype(float)
scores.index.name = "Kids_First_Biospecimen_ID"
if "Antigen_presentation_HLA_I" in scores and "Antigen_presentation_HLA_II" in scores:
    scores["HLA_I_minus_II"] = scores["Antigen_presentation_HLA_I"] - scores["Antigen_presentation_HLA_II"]
scores.to_csv(OUT / "step4c_ssGSEA_scores.tsv", sep="\t")
scores = scores.join(eco[["ecotype","cohort_group","location_class","age_dev_group"]], how="inner").dropna(subset=["ecotype"])
META_COLS = {"ecotype","cohort_group","location_class","age_dev_group"}
modules = [c for c in scores.columns if c not in META_COLS]
print(f"saved scores: {scores.shape};  modules: {len(modules)}")

saved scores: (349, 17);  modules: 13


## 3. Kruskal–Wallis + Dunn (3 ecotype) and KW (4 cohort)

In [4]:
def cliffs_delta(a,b):
    a=np.asarray(a); b=np.asarray(b); n1,n2=len(a),len(b)
    if n1==0 or n2==0: return np.nan
    return ((np.tile(a,(n2,1)).T > np.tile(b,(n1,1))).sum()
            - (np.tile(a,(n2,1)).T < np.tile(b,(n1,1))).sum())/(n1*n2)
def eps2(H,k,n): return max(0.0,(H-k+1)/(n-k)) if n-k>0 else np.nan

kw_rows, dunn_rows = [], []
for m in modules:
    grp = [scores.loc[scores.ecotype==e, m].dropna().values for e in ECOTYPE_ORDER]
    H,p = stats.kruskal(*grp)
    kw_rows.append({"module":m,"H":H,"p":p,"eps2":eps2(H,3,sum(map(len,grp))),
                    "median_Lymph":np.median(grp[0]),"median_Mye":np.median(grp[1]),"median_Desert":np.median(grp[2])})
    sub = scores[["ecotype",m]].dropna(); sub = sub[sub.ecotype.isin(ECOTYPE_ORDER)]
    dunn = sp.posthoc_dunn(sub, val_col=m, group_col="ecotype", p_adjust="fdr_bh")
    for i,g1 in enumerate(ECOTYPE_ORDER):
        for g2 in ECOTYPE_ORDER[i+1:]:
            dunn_rows.append({"module":m,"group1":g1,"group2":g2,
                              "q_BH":dunn.loc[g1,g2],
                              "cliffs_delta":cliffs_delta(scores.loc[scores.ecotype==g1,m].dropna(),
                                                          scores.loc[scores.ecotype==g2,m].dropna())})
kw = pd.DataFrame(kw_rows); kw["q_BH"]=multipletests(kw.p,method="fdr_bh")[1]
kw = kw.sort_values("eps2",ascending=False).reset_index(drop=True)
dunn_df = pd.DataFrame(dunn_rows)
kw.to_csv(OUT/"step4c_KW_by_ecotype.tsv",sep="\t",index=False)
dunn_df.to_csv(OUT/"step4c_Dunn_posthoc.tsv",sep="\t",index=False)

c_rows = []
for m in modules:
    g = [scores.loc[scores.cohort_group==c, m].dropna().values for c in COHORT_ORDER]
    H,p = stats.kruskal(*g)
    c_rows.append({"module":m,"H":H,"p":p,"eps2":eps2(H,4,sum(map(len,g))),
                   **{f"median_{c}":np.median(v) for c,v in zip(COHORT_ORDER,g)}})
ckw = pd.DataFrame(c_rows); ckw["q_BH"]=multipletests(ckw.p,method="fdr_bh")[1]
ckw = ckw.sort_values("eps2",ascending=False).reset_index(drop=True)
ckw.to_csv(OUT/"step4c_KW_by_cohort.tsv",sep="\t",index=False)
print("\n=== ECOTYPE KW (top) ==="); print(kw[["module","eps2","p","q_BH","median_Lymph","median_Mye","median_Desert"]].to_string(index=False))
print("\n=== COHORT KW (top) ===");  print(ckw[["module","eps2","p","q_BH"]].to_string(index=False))


=== ECOTYPE KW (top) ===
                     module     eps2            p         q_BH  median_Lymph  median_Mye  median_Desert
             TCM_Naive_loss 0.587259 2.774612e-45 3.606996e-44      0.401325    0.304693       0.226315
  Suppressive_cytokine_axis 0.575203 2.233722e-44 1.451920e-43      0.477220    0.413299       0.345383
             MDSC_monocytic 0.566585 9.919371e-44 4.298394e-43      0.532864    0.460953       0.360714
Antigen_presentation_HLA_II 0.488659 7.100399e-38 2.307630e-37      0.707875    0.652783       0.563061
            Exhaustion_core 0.482857 1.937441e-37 5.037346e-37      0.363980    0.253192       0.188864
        CCR2_chemokine_axis 0.470225 1.723028e-36 3.733226e-36      0.348221    0.195099       0.080758
             Treg_signature 0.453492 3.115164e-35 5.785305e-35      0.372457    0.285143       0.219035
          Senescence_T_cell 0.387480 2.839352e-30 4.613947e-30      0.505538    0.439853       0.379155
          MDSC_granulocytic 0.359018 3

## 4. Publication-ready boxplots (300 dpi) and mean-NES heatmap

In [5]:
def grid_box(df, group, order, palette, kw_df, fname, title):
    n_mod=len(modules); ncols=4; nrows=int(np.ceil(n_mod/ncols))
    fig,axes = plt.subplots(nrows,ncols,figsize=(ncols*3.0,nrows*2.6))
    axes=np.array(axes).reshape(nrows,ncols)
    for i,m in enumerate(modules):
        ax=axes[i//ncols,i%ncols]
        sns.boxplot(data=df, x=group, y=m, order=order, hue=group, palette=palette,
                    legend=False, ax=ax, fliersize=1.5, linewidth=0.7)
        ax.set_title(m,fontsize=9); ax.set_xlabel(""); ax.set_ylabel("ssGSEA NES",fontsize=8)
        ax.tick_params(axis='x', rotation=20, labelsize=7)
        row=kw_df[kw_df.module==m].iloc[0]
        ax.text(0.97,0.95,f"q={row.q_BH:.1e}\nε²={row.eps2:.2f}",
                ha="right",va="top",transform=ax.transAxes,fontsize=7,
                bbox=dict(boxstyle="round,pad=0.2",fc="white",ec="0.8",lw=0.5))
    for j in range(n_mod,nrows*ncols): axes[j//ncols,j%ncols].axis("off")
    fig.suptitle(title,fontsize=11,y=1.0); fig.tight_layout()
    fig.savefig(FIGDIR/f"{fname}.png",dpi=300,bbox_inches="tight")
    fig.savefig(FIGDIR/f"{fname}.pdf",bbox_inches="tight"); plt.close(fig)

grid_box(scores[scores.ecotype.isin(ECOTYPE_ORDER)], "ecotype", ECOTYPE_ORDER, ECOTYPE_COLORS,
         kw, "step4c_boxplot_ecotype", f"Extended ssGSEA modules vs ecotype (n={(scores.ecotype.isin(ECOTYPE_ORDER)).sum()})")
grid_box(scores[scores.cohort_group.isin(COHORT_ORDER)], "cohort_group", COHORT_ORDER, COHORT_COLORS,
         ckw, "step4c_boxplot_cohort", f"Extended ssGSEA modules vs cohort (n={(scores.cohort_group.isin(COHORT_ORDER)).sum()})")

# heatmap
em = scores[scores.ecotype.isin(ECOTYPE_ORDER)].groupby("ecotype")[modules].mean().T.loc[modules,ECOTYPE_ORDER].astype(float)
cm = scores[scores.cohort_group.isin(COHORT_ORDER)].groupby("cohort_group")[modules].mean().T.loc[modules,COHORT_ORDER].astype(float)
fig,axes = plt.subplots(1,2,figsize=(8.0,0.32*len(modules)+1.0))
sns.heatmap(em,annot=True,fmt=".2f",cmap="RdBu_r",center=0,cbar_kws={"label":"mean NES"},ax=axes[0],
            linewidths=0.3,linecolor="white",annot_kws={"size":7})
axes[0].set_title("Mean ssGSEA NES — ecotype"); axes[0].set_xlabel(""); axes[0].set_ylabel("")
sns.heatmap(cm,annot=True,fmt=".2f",cmap="RdBu_r",center=0,cbar_kws={"label":"mean NES"},ax=axes[1],
            linewidths=0.3,linecolor="white",annot_kws={"size":7})
axes[1].set_title("Mean ssGSEA NES — cohort"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
fig.tight_layout()
fig.savefig(FIGDIR/"step4c_heatmap.png",dpi=300,bbox_inches="tight")
fig.savefig(FIGDIR/"step4c_heatmap.pdf",bbox_inches="tight"); plt.close(fig)
print("figures saved to", FIGDIR)

figures saved to /sessions/blissful-gifted-dirac/mnt/Open PBTA/output/figs_step4c


## 5. Headline numbers — ecotype × module Dunn posthoc with Cliff's δ

In [6]:
head_dunn = dunn_df.copy()
head_dunn["abs_delta"] = head_dunn["cliffs_delta"].abs()
top = head_dunn.sort_values("abs_delta", ascending=False).head(15)
print(top.to_string(index=False))
print("\nHLA_I_minus_II mean per ecotype:")
print(scores.groupby("ecotype")["HLA_I_minus_II"].mean())

                     module              group1           group2         q_BH  cliffs_delta  abs_delta
             MDSC_monocytic Lymphocyte-inflamed    Immune-desert 4.883599e-44      0.973666   0.973666
             TCM_Naive_loss Lymphocyte-inflamed    Immune-desert 1.538068e-44      0.968122   0.968122
  Suppressive_cytokine_axis Lymphocyte-inflamed    Immune-desert 1.846524e-43      0.967198   0.967198
Antigen_presentation_HLA_II Lymphocyte-inflamed    Immune-desert 3.426945e-38      0.935320   0.935320
            Exhaustion_core Lymphocyte-inflamed    Immune-desert 7.346582e-35      0.917533   0.917533
        CCR2_chemokine_axis Lymphocyte-inflamed    Immune-desert 1.629487e-34      0.913606   0.913606
             Treg_signature Lymphocyte-inflamed    Immune-desert 1.432939e-34      0.903211   0.903211
          Senescence_T_cell Lymphocyte-inflamed    Immune-desert 1.005293e-29      0.875491   0.875491
          MDSC_granulocytic Lymphocyte-inflamed    Immune-desert 9.645920

## 6. Headline interpretation (Fecci / Sampson lens)

- **Exhaustion + Treg + Suppressive cytokine** axes all peak in Lymphocyte-inflamed → consistent with Grabowski 2021: TIL infiltration is necessary but not sufficient; the same tumors that recruit T cells also load them with inhibitory cues. Reframes "inflamed" as **"inflamed-but-exhausted"**, the natural rationale for ICI ± exhaustion-axis combination in pediatric glioma (cf. Sampson 2020 TIM3/LAG3 combo trials).
- **TCM/Naive loss** (largest ε²=0.587) tracks ecotype monotonically: Lymph > Mye > Desert. Maps to Grabowski's "T-cell senescence" domain — pediatric tumors do exhibit terminal-differentiation skew.
- **CSF1R-TAM axis** higher in Lymph/Mye than Desert: BLZ945-style monotherapy (Sampson 2020) is plausibly **least effective in the Desert ecotype** — translational nuance.
- **CCR2 chemokine axis** ε²=0.470, strongest separation Lymph vs Desert (δ=0.91): pinpoints Desert as the **chemokine-silent** TME and supports CCR2-axis adjunct as ecotype-specific hypothesis.
- **MDSC monocytic** ε²=0.567 (huge): Lymph and Mye both carry strong M-MDSC fingerprint; Desert is depleted. This re-casts "inflammation" in pediatric glioma as partly **M-MDSC-driven myeloid activation**, not pure T-cell inflammation.
- **HLA-I vs HLA-II axis**: Desert shows **higher HLA-I − HLA-II** (i.e. relative HLA-I retention with HLA-II loss) — matches the Grabowski 2021 description of glioma antigen-presentation reprogramming.
- **Cohort axis**: senescence and HLA-I/II offset are the strongest discriminators across DMG_K27 / DHG_G34 / pHGG_WT / IHG — DHG_G34 has lowest HLA-II and lowest MDSC scores, consistent with its enrichment in Desert ecotype.
